# Text Classification Task

Jaisa ki humne mention kiya, hum **AG_NEWS** dataset par based ek simple text classification task par focus karenge, jisme news headlines ko 4 categories mein classify karna hai: World, Sports, Business aur Sci/Tech.

## Dataset

Yeh dataset [`torchtext`](https://github.com/pytorch/text) module mein included hai, isliye hum ise aasaani se access kar sakte hain.

In [ ]:
import torch
import torchtext
import os
import collections
os.makedirs('./data',exist_ok=True)
train_dataset, test_dataset = torchtext.datasets.AG_NEWS(root='./data')
classes = ['World', 'Sports', 'Business', 'Sci/Tech']

Yahan, `train_dataset` aur `test_dataset` mein aise collections hain jo respectively class (class ki number) aur text ke pairs return karte hain, jaise ki:

In [ ]:
list(train_dataset)[0]

Toh, chalo apne dataset se pehli 10 nayi headlines print karte hain:

In [ ]:
for i,x in zip(range(5),train_dataset):
    print(f"**{classes[x[0]]}** -> {x[1]}")


Kyunki datasets iterators hote hain, agar hum data ko kai baar use karna chahte hain toh hume use list mein convert karna hoga:

In [ ]:
train_dataset, test_dataset = torchtext.datasets.AG_NEWS(root='./data')
train_dataset = list(train_dataset)
test_dataset = list(test_dataset)

## Tokenization

Ab hume text ko **numbers** mein badalna hoga, jinhe tensors ke roop mein represent kiya ja sake. Agar hume word-level representation chahiye, toh hume do cheezein karni hongi:
* **Tokenizer** use karke text ko **tokens** mein split karna
* Un tokens ki ek **vocabulary** banana.

In [ ]:
tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
tokenizer('He said: hello')

In [ ]:
counter = collections.Counter()
for (label, line) in train_dataset:
    counter.update(tokenizer(line))
vocab = torchtext.vocab.vocab(counter, min_freq=1)

Vocabulary use karke, hum apni tokenized string ko numbers ke set mein aasaani se encode kar sakte hain:

In [ ]:
vocab_size = len(vocab)
print(f"Vocab size if {vocab_size}")

stoi = vocab.get_stoi() # dict to convert tokens to indices

def encode(x):
    return [stoi[s] for s in tokenizer(x)]

encode('I love to play with my words')

## Bag of Words Text Representation

Kyunki words meaning ko represent karte hain, kabhi-kabhi hum sirf individual words ko dekhkar text ka meaning samajh sakte hain, chahe woh sentence mein kisi bhi order mein hon. Jaise, news classify karte waqt, *weather*, *snow* jaise words *weather forecast* ka sanket de sakte hain, jabki *stock*, *dollar* jaise words *financial news* ki taraf ishara karenge.

**Bag of Words** (BoW) vector representation sabse commonly use hone waali traditional vector representation hai. Har word ko ek vector index se link kiya jaata hai, aur vector element mein kisi diye gaye document mein us word ke occurrences ki number hoti hai.

![Yeh image dikhati hai ki Bag of Words ka vector representation memory mein kaise represent hota hai.](../../../../../lessons/5-NLP/13-TextRep/images/bag-of-words-example.png)

> **Note**: Aap BoW ko text mein individual words ke liye saare one-hot-encoded vectors ka sum bhi maan sakte hain.

Neeche Scikit Learn Python library use karke Bag of Words representation banane ka ek example diya gaya hai:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
vectorizer.fit_transform(corpus)
vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

AG_NEWS dataset ke vector representation se bag-of-words vector calculate karne ke liye, hum neeche diye gaye function ka use kar sakte hain:

In [ ]:
vocab_size = len(vocab)

def to_bow(text,bow_vocab_size=vocab_size):
    res = torch.zeros(bow_vocab_size,dtype=torch.float32)
    for i in encode(text):
        if i<bow_vocab_size:
            res[i] += 1
    return res

print(to_bow(train_dataset[0][1]))

> **Note:** Yahan hum global `vocab_size` variable use kar rahe hain taaki vocabulary ka default size specify kar sakein. Kyunki aksar vocabulary ka size kaafi bada hota hai, hum vocabulary size ko sabse zyada frequently use hone waale words tak limit kar sakte hain. `vocab_size` value ko kam karke neeche diya gaya code run karne ki koshish karein, aur dekhein ki yeh accuracy ko kaise affect karta hai. Aapko thodi accuracy drop ki ummeed rakhni chahiye, lekin performance improvement ke badle yeh dramatic nahi hoga.

## BoW Classifier ko Train karna

Ab jab humne apne text ka Bag-of-Words representation banana seekh liya hai, chalo uske upar ek classifier train karte hain. Sabse pehle, hume apne dataset ko training ke liye is tarah convert karna hoga ki saare positional vector representations ko Bag-of-Words representation mein badla ja sake. Yeh `bowify` function ko standard torch `DataLoader` mein `collate_fn` parameter ke roop mein pass karke kiya ja sakta hai:

In [ ]:
from torch.utils.data import DataLoader
import numpy as np 

# this collate function gets list of batch_size tuples, and needs to 
# return a pair of label-feature tensors for the whole minibatch
def bowify(b):
    return (
            torch.LongTensor([t[0]-1 for t in b]),
            torch.stack([to_bow(t[1]) for t in b])
    )

train_loader = DataLoader(train_dataset, batch_size=16, collate_fn=bowify, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, collate_fn=bowify, shuffle=True)

Ab chalo ek simple classification neural network define karte hain jisme ek linear layer hai. Input vector ka size `vocab_size` ke barabar hai, aur output size classes ki number (4) ke anuroop hai. Kyunki hum classification task solve kar rahe hain, final activation function `LogSoftmax()` hai.

In [ ]:
net = torch.nn.Sequential(torch.nn.Linear(vocab_size,4),torch.nn.LogSoftmax(dim=1))

Ab hum standard PyTorch training loop define karenge. Kyunki hamara dataset kaafi bada hai, teaching purpose ke liye hum sirf ek epoch ke liye train karenge, aur kabhi-kabhi ek epoch se bhi kam (training ko limit karne ke liye `epoch_size` parameter specify kiya jaata hai). Hum training ke dauraan accumulated training accuracy bhi report karenge; reporting ki frequency `report_freq` parameter se specify hoti hai.

In [ ]:
def train_epoch(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.NLLLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,features in dataloader:
        optimizer.zero_grad()
        out = net(features)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count

In [ ]:
train_epoch(net,train_loader,epoch_size=15000)

## Bigrams, Trigrams aur N-grams

Bag of Words approach ki ek limitation yeh hai ki kuch words multi-word expressions ka hissa hote hain. Jaise, 'hot dog' ka meaning bilkul alag hota hai, jabki 'hot' aur 'dog' words ka doosre contexts mein alag-alag meaning hota hai. Agar hum 'hot' aur 'dog' words ko hamesha ek hi vector se represent karein, toh yeh hamare model ko confuse kar sakta hai.

Is problem ko solve karne ke liye, **N-gram representations** ka use aksar document classification ke methods mein hota hai, jahan har word, bi-word ya tri-word ki frequency classifiers ko train karne ke liye ek useful feature hoti hai. Jaise, bigram representation mein, hum original words ke alawa saare word pairs ko vocabulary mein add karenge.

Neeche ek example diya gaya hai ki Scikit Learn use karke bigram Bag of Words representation kaise banaya ja sakta hai:

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2), token_pattern=r'\b\w+\b', min_df=1)
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
bigram_vectorizer.fit_transform(corpus)
print("Vocabulary:\n",bigram_vectorizer.vocabulary_)
bigram_vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()


N-gram approach ki main kami yeh hai ki vocabulary ka size bahut tezi se badhne lagta hai. Practice mein, hume N-gram representation ko kuch dimensionality reduction techniques, jaise *embeddings*, ke saath combine karna padta hai, jinhe hum agli unit mein discuss karenge.

Hamare **AG News** dataset mein N-gram representation use karne ke liye, hume ek special ngram vocabulary banani hogi:

In [ ]:
counter = collections.Counter()
for (label, line) in train_dataset:
    l = tokenizer(line)
    counter.update(torchtext.data.utils.ngrams_iterator(l,ngrams=2))
    
bi_vocab = torchtext.vocab.vocab(counter, min_freq=1)

print("Bigram vocabulary length = ",len(bi_vocab))

Hum upar diye gaye code se classifier ko train kar sakte hain, lekin woh memory ke lihaaz se bahut inefficient hoga. Agli unit mein, hum embeddings use karke bigram classifier ko train karenge.

> **Note:** Aap sirf un ngrams ko rakh sakte hain jo text mein specified number se zyada baar aate hain. Yeh ensure karega ki kam baar aane waale bigrams hata diye jaayein, aur dimensionality kaafi had tak kam ho jaaye. Aisa karne ke liye, `min_freq` parameter ko high value par set karein, aur vocabulary ki length mein badlaav ko observe karein.

## Term Frequency Inverse Document Frequency (TF-IDF)

BoW representation mein, words ki occurrences ko equal weight diya jaata hai, chahe woh word koi bhi ho. Lekin, yeh clear hai ki common words jaise *a*, *in* etc. classification ke liye utne important nahi hote jitne specialized words. Waise bhi, zyadatar NLP tasks mein kuch words doosre words se zyada relevant hote hain.

**TF-IDF** ka matlab hai **Term Frequency–Inverse Document Frequency**. Yeh Bag of Words ka ek variation hai, jisme document mein word ki presence dikhane waali 0/1 binary value ke bajaye ek floating-point value use hoti hai, jo corpus mein word ki occurrence ki frequency se related hoti hai.

Formally, kisi document $j$ mein word $i$ ka weight $w_{ij}$ is tarah define hota hai:
$$
w_{ij} = tf_{ij}\times\log({N\over df_i})
$$
jahan:
* $tf_{ij}$ document $j$ mein word $i$ ke occurrences ki number hai, yaani woh BoW value jo humne pehle dekhi thi
* $N$ collection mein documents ki total number hai
* $df_i$ poore collection mein word $i$ ko contain karne waale documents ki number hai

TF-IDF value $w_{ij}$ document mein word ke occurrences ki number ke proportion mein badhti hai aur corpus mein us word ko contain karne waale documents ki number se adjust hoti hai. Yeh is fact ko balance karne mein help karta hai ki kuch words doosre words se zyada baar aate hain. Jaise, agar koi word *har* document mein aata hai, toh $df_i=N$, aur $w_{ij}=0$, aur aise words ko poori tarah ignore kar diya jaayega.

Aap Scikit Learn use karke aasaani se text ka TF-IDF vectorization bana sakte hain:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))
vectorizer.fit_transform(corpus)
vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

## Conclusion

Haalaanki TF-IDF representations alag-alag words ko frequency weights dete hain, woh meaning ya order ko represent karne mein asamarth hain. Jaisa ki mashhoor linguist J. R. Firth ne 1935 mein kaha tha, "kisi word ka poora meaning hamesha contextual hota hai, aur context se alag meaning ki koi bhi study seriously nahi li ja sakti." Hum course mein aage seekhenge ki language modeling use karke text se contextual information kaise capture karein.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->